In [33]:
import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from Models import CNN_1

In [34]:
batch_size_=32
num_epochs = 50

In [35]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [36]:
train_data_full = datasets.ImageFolder(root='./DATASET/TRAIN', transform=train_transform)
val_data_full   = datasets.ImageFolder(root='./DATASET/VAL', transform=val_test_transform)
test_data_full = datasets.ImageFolder(root='./DATASET/TEST', transform=val_test_transform)


In [37]:
train_dataset=DataLoader(train_data_full, batch_size=batch_size_, shuffle=True)
val_dataset=DataLoader(val_data_full, batch_size=batch_size_)
Test_dataset=DataLoader(test_data_full, batch_size=batch_size_)

In [38]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = CNN_1().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)

In [39]:

for epoch in range(num_epochs):
    model1.train()
    total_loss = 0
    for images, labels in train_dataset:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model1(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Total loss: {total_loss:.4f}")


Epoch 1, Loss: 880.0944


In [40]:
model1.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_dataset:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model1(images)
        preds = (outputs > 0.5).int().squeeze()

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.8231382978723404


In [41]:
model1.eval()

with torch.no_grad():
    for images, labels in Test_dataset:
        images = images.to(device)

        outputs = model1(images)
        preds = (outputs > 0.5).int()

        print(preds)
        break

tensor([[0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0],
        [0]], device='cuda:0', dtype=torch.int32)
